In [2]:
import pandas as pd
import numpy as np

In [4]:
pima = pd.read_csv("Pima Indians Diabetes.csv")
brfss = pd.read_csv("BRFSS Diabetes Indicators.csv")
diabd = pd.read_csv("DiaBD_A Diabetes Dataset for Enhanced Risk Analysis and Research in Bangladesh.csv")

In [5]:
# 2) Prepare Pima dataset

pima = pima[['Age', 'BMI', 'Glucose', 'BloodPressure', 'Outcome']].copy()

# Create HighBP from BloodPressure
pima['HighBP'] = np.where(pima['BloodPressure'] >= 90, 1, 0)

# Add Source
pima['Source'] = 'Pima'

In [6]:
brfss = brfss[['Age', 'BMI', 'HighBP', 'Diabetes_012']].copy()

# Remove prediabetes rows
brfss = brfss[brfss['Diabetes_012'] != 1]

# Convert Diabetes_012 to binary Outcome
brfss['Outcome'] = brfss['Diabetes_012'].replace({
    0: 0,
    2: 1
})

In [7]:
brfss['Glucose'] = np.nan
brfss['BloodPressure'] = np.nan

# Add Source
brfss['Source'] = 'BRFSS'

# Keep needed columns only
brfss = brfss[['Age', 'BMI', 'Glucose', 'BloodPressure', 'HighBP', 'Outcome', 'Source']]

In [8]:
diabd = diabd[['age', 'bmi', 'glucose', 'diastolic_bp', 'diabetic']].copy()

# Rename columns
diabd = diabd.rename(columns={
    'age': 'Age',
    'bmi': 'BMI',
    'glucose': 'Glucose',
    'diastolic_bp': 'BloodPressure',
    'diabetic': 'Outcome'
})

# Convert diabetic to binary
diabd['Outcome'] = diabd['Outcome'].replace({
    'Yes': 1,
    'No': 0,
    'yes': 1,
    'no': 0,
    1: 1,
    0: 0
})

# Create HighBP from BloodPressure
diabd['HighBP'] = np.where(diabd['BloodPressure'] >= 90, 1, 0)

# Add Source
diabd['Source'] = 'DiaBD_A'

/tmp/ipykernel_1418/3502417515.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  diabd['Outcome'] = diabd['Outcome'].replace({


In [9]:
final_columns = [
    'Age',
    'BMI',
    'Glucose',
    'BloodPressure',
    'HighBP',
    'Outcome',
    'Source'
]

pima = pima[final_columns]
brfss = brfss[final_columns]
diabd = diabd[final_columns]

In [11]:
# combine dataset

combined = pd.concat(
    [pima, brfss, diabd],
    ignore_index=True
)

In [12]:
numeric_columns = [
    'Age',
    'BMI',
    'Glucose',
    'BloodPressure',
    'HighBP',
    'Outcome'
]

for col in numeric_columns:
    combined[col] = pd.to_numeric(combined[col], errors='coerce')

In [13]:
combined = combined.dropna(subset=['Age', 'BMI', 'HighBP', 'Outcome'])

In [14]:
# Reset index
combined = combined.reset_index(drop=True)
# 9. Save final dataset
combined.to_csv("combined_diabetes_dataset.csv", index=False)

In [15]:
print(combined.head())
print(combined.shape)
print(combined.info())
print(combined['Source'].value_counts())
print(combined.isnull().sum())

    Age   BMI  Glucose  BloodPressure  HighBP  Outcome Source
0  50.0  33.6    148.0           72.0     0.0      1.0   Pima
1  31.0  26.6     85.0           66.0     0.0      0.0   Pima
2  32.0  23.3    183.0           64.0     0.0      1.0   Pima
3  21.0  28.1     89.0           66.0     0.0      0.0   Pima
4  33.0  43.1    137.0           40.0     0.0      1.0   Pima
(255105, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255105 entries, 0 to 255104
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Age            255105 non-null  float64
 1   BMI            255105 non-null  float64
 2   Glucose        6056 non-null    float64
 3   BloodPressure  6056 non-null    float64
 4   HighBP         255105 non-null  float64
 5   Outcome        255105 non-null  float64
 6   Source         255105 non-null  object 
dtypes: float64(6), object(1)
memory usage: 13.6+ MB
None
Source
BRFSS      249049
DiaBD_A      5288
